# Minimum Jumps to Reach the End (with All Paths)

We are given an array $arr[0..n-1]$ where each element $arr[i]$ denotes the **maximum number of steps** that can be taken forward from index $i$.

Goal: 
1. Find the minimum number of jumps to reach the last index.
2. Print **all paths** that use the minimum number of jumps.

---

## Dynamic Programming Phase

Let $dp[i]$ = minimum number of jumps needed to reach the end from index $i$.

### Base Case:
\[
dp[n-1] = 0 \quad \text{(already at the end)}
\]

### Transition:
\[
dp[i] = 1 + \min_{1 \leq j \leq arr[i]} \{ dp[i+j] \}, \quad \text{if $i+j < n$ and $dp[i+j]$ exists}
\]

If no valid move exists:
\[
dp[i] = \infty
\]

---

## Path Reconstruction Phase

We perform BFS starting from index $0$:
- Each node is a tuple $(i, steps, jumps, path)$
  - $i$ = current index
  - $steps$ = maximum jumps from this index
  - $jumps$ = minimum jumps remaining from this index ($dp[i]$)
  - $path$ = path so far

Whenever we reach $dp[i] = 0$ (last index), we print the path.


In [4]:
from collections import deque
from dataclasses import dataclass
from typing import List, Optional, Tuple

@dataclass
class Pair:
    i: int        # current index
    s: int        # arr[i] steps allowed
    j: Optional[int]  # dp[i] (jumps left to reach end), None if unreachable
    psf: str      # path-so-far, like "0->1->4"

def all_paths_min_jumps_detailed(arr: List[int]) -> Tuple[List[Optional[int]], List[str]]:
    """
    Compute dp[] = min jumps from each index to end (or None if unreachable),
    then BFS-enumerate all paths from 0 to end that use the minimum number of jumps.
    Print detailed step-by-step reasoning for both phases.
    Returns (dp, paths).
    """
    n = len(arr)
    dp: List[Optional[int]] = [None] * n
    dp[-1] = 0  # base: last index needs 0 jumps

    print("\n=== INPUT ===")
    print("Indices : ", list(range(n)))
    print("Array   : ", arr)
    print()

    # DP PHASE
    print("=== DP PHASE: compute dp[i] = min jumps from i to end ===")
    print(f"Initialize dp with dp[{n-1}] = 0 (last index).")
    print("dp:", dp)
    print()

    for i in range(n - 2, -1, -1):  # from second-last down to 0
        print(f"Compute dp[{i}] (arr[{i}] = {arr[i]}):")
        steps = arr[i]
        if steps == 0:
            # cannot move from here
            print(f"  arr[{i}] == 0 -> cannot move forward. dp[{i}] stays None.")
            print()
            continue

        candidates = []  # list of (target_index, dp[target_index])
        for step in range(1, steps + 1):
            ci = i + step
            if ci >= n:
                print(f"  step {step}: i+{step} = {ci} -> out of bounds (skip)")
                continue
            print(f"  step {step}: consider jump to index {ci} (dp[{ci}] = {dp[ci]})")
            if dp[ci] is not None:
                candidates.append((ci, dp[ci]))

        if not candidates:
            print(f"  -> No reachable index from {i} leads to the end (all candidates None). dp[{i}] = None")
            dp[i] = None
        else:
            # pick minimal dp among candidates (worst-case remaining jumps)
            min_dp_val = min(val for (_, val) in candidates)
            # show all indices that achieve min_dp_val (there can be multiple)
            winners = [ci for (ci, val) in candidates if val == min_dp_val]
            dp[i] = min_dp_val + 1
            print(f"  candidates (index, dp): {candidates}")
            print(f"  minimal dp among candidates = {min_dp_val} at indices {winners}")
            print(f"  therefore dp[{i}] = {min_dp_val} + 1 = {dp[i]}")
        print("  dp now:", dp)
        print()

    print("Completed DP phase.")
    print("Final dp array (min jumps from each index, None = unreachable):")
    print(dp)
    print()

    # BFS PHASE (enumerate all minimum-jump paths)
    print("=== BFS PHASE: enumerate all paths that achieve dp[0] ===")
    if dp[0] is None:
        print("Start index 0 cannot reach the end (dp[0] is None). No paths.")
        return dp, []

    q = deque()
    q.append(Pair(0, arr[0], dp[0], "0"))   # start from index 0
    paths: List[str] = []
    step_no = 0

    while q:
        step_no += 1
        # print queue snapshot
        queue_snapshot = [(p.i, p.s, p.j, p.psf) for p in q]
        print(f"[BFS step {step_no}] Queue before pop: {queue_snapshot}")
        rem = q.popleft()
        print(f"  Pop -> index={rem.i}, stepsAllowed={rem.s}, jumpsLeft(dp)={rem.j}, path='{rem.psf}'")

        if rem.j == 0:
            # we've reached an index that requires 0 jumps -> it's the last segment of a min path
            print(f"    rem.j == 0 -> reached end via minimum jumps. Path complete: {rem.psf}.")
            paths.append(rem.psf + ".")
            print()
            continue

        # try all next steps from this index
        for step in range(1, rem.s + 1):
            ci = rem.i + step
            if ci >= n:
                print(f"    try step {step}: index {ci} out of bounds -> skip")
                continue
            needed = rem.j - 1
            print(f"    try step {step}: move to {ci}; dp[{ci}] = {dp[ci]}; needed = {needed}")
            # we only follow edges that keep us on a minimum-jump route:
            # dp[ci] must exist and must equal rem.j - 1
            if dp[ci] is not None and dp[ci] == needed:
                new_psf = rem.psf + "->" + str(ci)
                q.append(Pair(ci, arr[ci], dp[ci], new_psf))
                print(f"      -> enqueue index={ci}, stepsAllowed={arr[ci]}, jumpsLeft={dp[ci]}, path='{new_psf}'")
            else:
                print(f"      -> do NOT enqueue (either dp[{ci}] is None or dp[{ci}] != {needed})")
        print()

    print("BFS finished.")
    print(f"All minimum-jump paths from 0 to end (count = {len(paths)}):")
    for p in paths:
        print(" ", p)
    print()
    return dp, paths


# Example usage:
if __name__ == "__main__":
    arr = [3, 3, 0, 2, 1, 2, 4, 2, 0, 0]
    dp, paths = all_paths_min_jumps_detailed(arr)



=== INPUT ===
Indices :  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Array   :  [3, 3, 0, 2, 1, 2, 4, 2, 0, 0]

=== DP PHASE: compute dp[i] = min jumps from i to end ===
Initialize dp with dp[9] = 0 (last index).
dp: [None, None, None, None, None, None, None, None, None, 0]

Compute dp[8] (arr[8] = 0):
  arr[8] == 0 -> cannot move forward. dp[8] stays None.

Compute dp[7] (arr[7] = 2):
  step 1: consider jump to index 8 (dp[8] = None)
  step 2: consider jump to index 9 (dp[9] = 0)
  candidates (index, dp): [(9, 0)]
  minimal dp among candidates = 0 at indices [9]
  therefore dp[7] = 0 + 1 = 1
  dp now: [None, None, None, None, None, None, None, 1, None, 0]

Compute dp[6] (arr[6] = 4):
  step 1: consider jump to index 7 (dp[7] = 1)
  step 2: consider jump to index 8 (dp[8] = None)
  step 3: consider jump to index 9 (dp[9] = 0)
  step 4: i+4 = 10 -> out of bounds (skip)
  candidates (index, dp): [(7, 1), (9, 0)]
  minimal dp among candidates = 0 at indices [9]
  therefore dp[6] = 0 + 1 = 1
  dp now: